# BPI Challenge 2015: permit applications across five municipalities

Five municipalities provide distinct logs of environmental-permit handling. The analysis asks where application information, administrative recording and decision dispatch create avoidable delay, while preserving lawful assessment, consultation and objection rights.

The questions below concern the complete recorded case. WISE evaluates the declared constraints and returns the numerical summaries, priorities and evidence. Figures display those returned results. The review scales are exploratory inputs awaiting process-owner approval; a flagged case is a starting point for investigation.

## PM² route

The work follows the six stages of [PM²](https://vdaalst.com/publications/p824.pdf). Revisit earlier stages when evidence changes the question or its scope.

| Stage | Work recorded here | Decision required |
|---|---|---|
| 1. Planning | Business goal, case unit, interpretation questions and candidate norms | Confirm the investigation scope and acceptable service outcomes |
| 2. Extraction | Read the supplied challenge logs and identify source versions | Check the extraction boundary and missing records |
| 3. Data processing | Retain whole cases, review activity bindings and context | Resolve ambiguous meanings and episode boundaries |
| 4. Mining and analysis | Review severity, coverage, context, priorities and original traces | Identify a recurring mechanism supported by evidence |
| 5. Evaluation | Compare views, settings and plausible explanations | Decide whether a proposed change warrants a trial |
| 6. Process improvement and support | Assign an owner, define a pilot and protect essential work | Measure the result before expanding the change |

The execution journal records the analytical run. The review journal near the end records business judgements separately; running a notebook does not approve a norm or complete a process-improvement trial.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import wise

try:
    from bpic_analysis import (
        challenge, summary, norm_table, criterion_summary, activity_summary,
        context_profiles, priorities, plot_overview, case_evidence,
        threshold_sensitivity, view_agreement, integrity, save_outputs, journal,
    )
except ModuleNotFoundError as error:
    if error.name != "bpic_analysis":
        raise
    from notebooks.bpic_analysis import (
        challenge, summary, norm_table, criterion_summary, activity_summary,
        context_profiles, priorities, plot_overview, case_evidence,
        threshold_sensitivity, view_agreement, integrity, save_outputs, journal,
    )

if wise.__version__ != "1.0.1":
    raise RuntimeError("Select the environment with the frozen 1.0.1 package.")

YEAR = 2015
data_root = os.environ.get("WISE_PAPER_DATA")
SAVE_OUTPUTS = False


def show_evidence(evidence):
    for label, value in evidence.items():
        display(Markdown("**" + label.replace("_", " ") + "**"))
        display(value)


def show_figures(run):
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)

## 1. Planning: the business question

**Goal:** provide clearer application handling and fewer avoidable completeness requests without weakening legal assessment or applicants' rights.

**Case unit:** one permit application within its municipality, retaining main-procedure and objection events. Municipality is part of the case identity; similarly named case identifiers in different logs are not one case.

Investigate:

1. Where does the first recorded acknowledgment, decision or dispatch interval warrant review within a comparable procedure?
2. Do repeated completeness-subcase or decision-creation records describe avoidable repeated requests, different required subcases, or lawful revisions?
3. Which differences persist within each municipality after procedure and completeness context are considered?

Each municipality has its own population reference. The notebook keeps the five runs separate. Their relative priority values are not a performance league table.

In [ ]:
runs = challenge(YEAR, data_folder=data_root)
display(summary(runs))

for identifier, run in runs.items():
    if run["record"]["status"] != "computed":
        display(run["record"])
        raise RuntimeError("Resolve the recorded source or activity-binding issue before analysis.")
    display(Markdown("### " + identifier))
    display(run["specification"]["business_goal"])

## 2. Extraction: establish the evidence boundary

Read the five canonical municipality XES files as five sources. A copied XML export is an alternate representation, not a sixth municipality.

The selected codes were checked against their English labels across all five sources. Several other codes have ambiguous or different meanings. In particular, `01_HOOFD_020` can mean either reception through OLO or sending confirmation of receipt. The acknowledgment criterion therefore binds to `01_HOOFD_030_1`. Keep `activityNameEN` and `activityNameNL` available in source drill-down.

Dates such as `time:timestamp`, `dateFinished`, `dueDate`, `dateStop` and trace-level `endDatePlanned` describe different source concepts. Their equivalence has not been established. Municipality 2 does not provide the same trace-level planned-end field as the other four.

Source fingerprints and dataset counts appear in the execution summary. The integrity record below checks numerical norm round-trip agreement, signed decomposition, case-index agreement and the count of events retained by the log object. It is not an independent certification of raw-source completeness or event-by-event retention. Criterion-level availability appears later. These numerical checks do not establish legal and administrative correctness.

In [ ]:
integrity_records = {}
for identifier, run in runs.items():
    integrity_records[identifier] = integrity(run)
    display(Markdown("### " + identifier))
    display(integrity_records[identifier])

## 3. Data processing: preserve whole-case context

Begin within municipality and `caseProcedure`, retaining unknown values. Then inspect `requestComplete` and application `parts`. The completeness property may be retrospective; it must not be treated as information already available when an application arrived.

A case can include multiple completeness subcases and objections. Preserve those records when grouping. The trace's `termName` can refer to an objection or appeal term; comparing it indiscriminately with a main decision date could assess the wrong legal clock. A final event or recorded end date alone does not establish final resolution of every legal subprocedure.

Read the profile of missing and mixed context before comparing groups. The case-start and last-observed month labels use the earliest and latest usable recorded timestamps after UTC normalisation. They can differ from business-local calendar months and do not verify the actual process start or finish. Verified completion needs a reviewed terminal-state definition. The observed span is the interval between these recorded endpoints; it does not establish processing time. Calendar selections must retain all events of each selected case.

In [ ]:
context_tables = {}
for identifier, run in runs.items():
    context_tables[identifier] = context_profiles(run, attributes=['caseProcedure', 'requestComplete', 'parts', 'termName', 'caseStatus', 'last_phase'])
    display(Markdown("### " + identifier))
    display(context_tables[identifier])

### Turn goals into inspectable review criteria

The executable baseline examines submission-to-acknowledgment, submission-to-first-main-decision and first-decision-to-recorded-dispatch intervals. It also screens repeated decision creation and completeness-subcase treatment.

The timing inputs are exploratory review ranges. A universal duration limit would mix regular and extended procedures, suspensions, extensions and different recorded clocks. Procedure-specific statutory assessment remains pending until historical policy and source-date meanings have been reviewed.

Treat a repeated completeness-subcase record as work to explain. It may be a different required part of the application. A revised decision can protect an applicant's rights.

The table exposes the exact labels, scales, applicability, missing-data policy and stakeholder views used in the run. For a lag, the baseline uses the earliest usable start and the first usable end at or after it. It does not pair every cycle. Missing endpoints are unavailable evidence. Occurrence criteria count recorded event rows; they do not prove that repeated work was unnecessary. The Balanced view assigns equal total weight to each declared layer.

In [ ]:
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    display(norm_table(run))
    display(Markdown("**Questions awaiting additional semantics or policy evidence**"))
    display(run["specification"].get("pending_checks", []))

## 4. Mining and analysis: distinguish activity volume from case burden

Contrast registered submissions, acknowledgment, decision creation, recorded decision dates, dispatch dates and completeness-subcase treatment. The semantic mapping determines the analysis. A code prefix or numeric suffix alone does not establish a process phase or a required order.

The first measure counts recorded occurrences. The second counts cases with at least one occurrence. Their units differ. Neither is an estimate of staff effort, avoidable cost or a causal explanation.

In [ ]:
activity_tables = {}
for identifier, run in runs.items():
    table = activity_summary(run, labels=['01_HOOFD_010', '01_HOOFD_030_1', '01_HOOFD_490_1', '01_HOOFD_490_3', '01_HOOFD_490_5', '01_HOOFD_510_2', '01_HOOFD_510_3', '01_HOOFD_510_4', '01_HOOFD_110', '01_HOOFD_110_1'])
    activity_tables[identifier] = table
    display(Markdown("### " + identifier))
    display(table)
    if not table.empty:
        figure, axes = plt.subplots(1, 2, figsize=(14, 5), squeeze=False)
        table.set_index("activity")[["occurrences"]].plot.barh(
            ax=axes[0, 0], legend=False, color="C0"
        )
        table.set_index("activity")[["observed_cases"]].plot.barh(
            ax=axes[0, 1], legend=False, color="C1"
        )
        axes[0, 0].set_title("Recorded occurrences")
        axes[0, 1].set_title("Cases with an occurrence")
        axes[0, 0].set_xlabel("Event rows")
        axes[0, 1].set_xlabel("Whole cases")
        for axis in axes[0]:
            axis.xaxis.set_major_locator(MaxNLocator(5))
        figure.suptitle(identifier)
        figure.tight_layout()
        display(figure)
        plt.close(figure)

### Severity and evidence availability answer different questions

Look for a timing criterion with poor endpoint coverage before interpreting its mean severity. An unfinished or partially recorded application may lack a decision or dispatch endpoint. The baseline leaves this timing evidence unavailable; it does not convert endpoint absence into a legal failure.

Compare completeness-subcase treatment with decision-creation repetition. They concern different work and require different improvement mechanisms.

Inspect evaluated and in-scope counts beside mean severity. Evidence coverage is the fraction of in-scope cases that support a criterion evaluation. A low severity with little available evidence is inconclusive. A high severity under an illustrative scale still needs its process meaning checked.

**Measured baseline in the supplied logs.** Municipality 4 supplies evaluable acknowledgment evidence for 125 of 1,053 cases under the selected binding; the corresponding availability exceeds 92% in each of the other four logs. This difference calls for review of source-specific acknowledgment vocabulary and recording coverage before any delay comparison. It is not a count of applications that received no acknowledgment.

Procedure labels are absent for most cases in these extracts. This limits comparisons within known regular and extended procedures; the missing group is not a confirmed procedure class. The next step is to validate the source meaning and populate a reviewed whole-case procedure mapping where the original evidence supports one.

These findings describe the measured sources and norm snapshots. Their fingerprints are printed beside the tables below; rerun after input or norm changes.

Per-criterion coverage in this table is an unweighted fraction of in-scope cases. The source witness also shows each view's weighted case-level evidence coverage, which measures the available share of its in-scope criterion weight. Scores renormalise the usable evidence, so a high returned score with incomplete coverage leaves parts of the declared norm unassessed. Neither coverage measure certifies source completeness.

In [ ]:
criterion_tables = {}
for identifier, run in runs.items():
    table = criterion_summary(run)
    criterion_tables[identifier] = table
    display(Markdown("### " + identifier))
    display({"source_sha256": run["record"]["source_sha256"],
             "norm_fingerprint": run["record"]["norm_fingerprint"]})
    display(table)
    if not table.empty:
        axis = table.set_index("criterion_id")[["evidence_coverage", "mean_severity"]].plot.bar(
            figsize=(10, 4), ylim=(0, 1), rot=0
        )
        axis.set_ylabel("Fraction or mean severity")
        axis.set_title(identifier + ": evidence and review severity")
        axis.figure.tight_layout()
        display(axis.figure)
        plt.close(axis.figure)

### Choose a context for the investigation list

Use the declared procedure-and-completeness grouping within each municipality, then inspect the alternative whole-case groupings. A group containing unknown procedure values deserves data interpretation before a performance comparison.

Timing and Rework views may nominate different applications or contexts. Retain both questions: a long administrative interval and a repeated information request need not have the same cause.

Read the returned grouping and view specification beside each list. Priority combines case volume and a score gap relative to the population of this run. A zero relative priority does not establish an acceptable absolute result. Small-slice shrinkage is a declared sensitivity setting, not a fitted service target.

In [ ]:
investigation_lists = {}
for identifier, run in runs.items():
    investigation_lists[identifier] = {}
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=0)
        adjusted = priorities(run, view=view, gamma=20)
        investigation_lists[identifier][view] = backlog
        display(Markdown("### " + identifier + " / " + view))
        display(backlog.spec.to_dict())
        display(backlog.table.head(10))
        display(Markdown("**Same population with small-slice shrinkage**"))
        display(adjusted.table.head(10))
    for grouping in ['caseProcedure', 'requestComplete', 'parts']:
        backlog = priorities(run, by=grouping, gamma=0)
        display(Markdown("**Alternative whole-case grouping: " + str(grouping) + "**"))
        display(backlog.spec.to_dict())
        display(backlog.table.head(10))

### Read the overview together with its signed explanation

Read each municipality panel against its own reference population. A strong Rework contribution can coexist with a favourable Timing contribution. Such a case may have more recorded assessment work while reaching a decision relatively quickly. This is a prompt for review, not evidence that thorough assessment should be shortened.

These figures use the evaluated cases and library-returned priorities and contributions. Positive layer contributions add to the relative penalty gap; negative contributions offset it. Preserve both signs when explaining a shortlisted group. The decomposition describes the score, not the cause of a process problem.

In [ ]:
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    show_figures(run)

### Return to the actual case

For every municipality, inspect one witness of repeated completeness-subcase treatment and one witness of the submission-to-first-decision interval. Check the English labels, application parts, procedure, requests, dated responses and objection records. Separate repeated requests for the same missing information from different subcases.

The helper chooses a recorded witness for the requested criterion and returns the original source events, case context and evaluated criterion evidence. The full source trace remains available. Read rows with tied timestamps as unresolved ordering unless the source provides a stronger order. To inspect another case, pass its actual identifier as `case_id` to `case_evidence`.

In [ ]:
for municipality in range(1, 6):
    identifier = "bpic2015_" + str(municipality)
    run = runs[identifier]
    for suffix in ("05", "02"):
        criterion = "M" + str(municipality) + "_" + suffix
        display(Markdown("### " + identifier + " / source witness: " + criterion))
        show_evidence(case_evidence(run, criterion=criterion))

### Place the case evidence on a timeline

This figure uses event timestamps returned by the library for a complete recorded case selected by the first diagnostic criterion. Named lanes highlight norm activities; the grey lane keeps the remaining recorded activities visible. Coincident points can overlap and establish no order. Undated events remain in the source table and cannot be placed on the time axis. The selected case is an investigation example, not a representative sample; return to its full source witness before interpreting an apparent gap.

In [ ]:
from bpic_analysis import plot_case_timeline
for identifier, run in runs.items():
    if "result" not in run:
        continue
    figure = plot_case_timeline(run)
    display(figure)
    plt.close(figure)

## 5. Evaluation: challenge the interpretation

Cross-municipality claims require comparable procedure classes, legal periods and activity meanings. The present runs provide separate diagnostic baselines, not adjusted estimates of municipal effectiveness.

Before reporting an avoidable delay, establish the relevant clock and any suspension or extension. Before reporting rework, establish whether two records concern the same request, a different application part or a justified revision. A coarse event sequence does not establish either conclusion.

The factors below scale both the lag threshold `delta` and saturation width `width`. For occurrence criteria they scale the saturation width `K` while keeping the integer occurrence tolerance `k` fixed. The helper rescores each declared scenario with WISE. These tests examine the chosen scales; they do not estimate an optimal policy. The view agreement table compares positive-priority shortlist membership under the declared views; it does not measure causal confidence or validate a business policy. Inspect a large disagreement by returning to its evidence.

In [ ]:
sensitivity_tables = {}
view_tables = {}
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    sensitivity_tables[identifier] = threshold_sensitivity(run, factors=(0.5, 1, 2))
    view_tables[identifier] = view_agreement(run, k=10)
    display(sensitivity_tables[identifier])
    display(view_tables[identifier])

## 6. Process improvement and support: define a feasible trial

| Confirmed mechanism required | Feasible change and owner | Pilot measure | Protected outcome |
|---|---|---|---|
| The same missing information is requested repeatedly by different reviewers | Permit-service manager trials a consolidated request, confirmation and named case owner for one procedure | Repeat requests for the same information and verified waiting for a response | Complete technical assessment and clear applicant communication |
| Dispatch waits after an authorised decision because ownership is unclear | Administrative lead trials a decision-to-dispatch checklist and queue owner | Verified administrative waiting after decision authorisation | Correct publication, notification and appeal information |
| Handoffs lose the current application status | Case-management lead trials a shared status and next-action record | Unanswered applicant queries and avoidable status corrections | Lawful consultation, review and objection handling |

Choose one municipality and one comparable procedure for the first pilot. Review actual case documents before changing requests or dispatch work. Define the historical and prospective cohorts under the same approved rules.

A change proceeds after the mechanism has been confirmed in source-record review and its owner has accepted the scope. Declare the baseline cohort, observation horizon, exceptions, outcome measures and stop conditions before the pilot. Record changes in case mix and evidence collection. A shorter recorded trace or lower diagnostic score alone does not establish a better process.

### Full analysis journal and review record

The execution journal below covers the run and its evidence checks. Add dated observations to `review_journal` as cases are discussed. Each entry should identify its supporting case or table, a plausible competing explanation, the reviewer decision and the next action. Empty fields remain unfilled work; they are not invented findings.

In [ ]:
execution_journal = journal(runs)
display(execution_journal)

review_journal = {
    "business_goal_and_population": "",
    "norms_approved_or_changed": "",
    "evidence_identifiers_reviewed": [],
    "observed_pattern": "",
    "competing_explanations": [],
    "confirmed_avoidable_mechanism": "",
    "decision_and_owner": "",
    "pilot_scope_and_comparison": "",
    "protected_outcomes_and_stop_conditions": [],
    "review_date": "",
    "next_action": "",
}
display(review_journal)

### Decision gates before reporting an improvement

- Are municipality and case identity retained throughout the analysis?
- Do activity codes mean the same thing in the cases being compared?
- Has the applicable procedure, legal clock and suspension history been established?
- Does the proposed change target a confirmed administrative mechanism while retaining assessment and objection rights?

The portable output option saves the actual run records, tables and figures in `outputs`. Review case-level information before sharing saved outputs. Cell outputs, when present, describe the last saved execution. Rerun all cells after input or norm changes.

In [ ]:
if SAVE_OUTPUTS:
    output_folder = Path("outputs") / "bpic2015"
    save_outputs(runs, output_folder)
    display({"saved_outputs": output_folder.as_posix()})

## Source and method notes

- [BPI Challenge 2015 process description and municipality sources](https://ais.win.tue.nl/bpi/2015/challenge.html).
- English and Dutch activity labels, trace attributes and event dates in the five supplied XES logs.
- [PM² methodology](https://vdaalst.com/publications/p824.pdf).

The declared norm and source fingerprints are recorded by the helper. WISE calculates constraint evaluations, scores, priorities and decompositions; notebook code selects and presents those results. Pending rules require the additional evidence stated in the norm specification.